# Task 3: rooftop solar for a campus building

**Python for Energy System Modelling: workshop for postgraduate students in energy and sustainable engineering**
Ammar Ahmed Wafad

How much of a real university building's electricity could rooftop solar supply, and how much CO2 would it save? In Tasks 1 and 2 you worked with hourly data. Here you work with **monthly** data from the university's energy invoices, which is what most organisations actually have.

- **Duration:** about 15 minutes. Keep the same group roles as in Tasks 1 and 2.
- **Data:** monthly electricity use of 41 buildings on one university campus, August 2021 to July 2022, from energy invoices. **The data is shared for this workshop only. Please do not publish or share it.**
- **Outputs:** figures and tables are saved in the `outputs/` folder.

## 1. Setup

Keep this notebook in the same folder as the `data` folder, which must contain `campus_electricity_2021_22.csv`. Run the cell below. It needs no installation.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display

DATA_DIR = Path("data")
FIG_DIR = Path("outputs/figures")
TABLE_DIR = Path("outputs/tables")
for folder in (DATA_DIR, FIG_DIR, TABLE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

COLOURS = {"demand": "#1F2933", "solar": "#E8A202", "export": "#9AA5B1", "used": "#6E9F1A"}
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#E4E7EB", "legend.frameon": False,
})


def save_figure(fig, name):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print(f"Figure saved: {path}")


def save_table(df, name):
    path = TABLE_DIR / f"{name}.csv"
    df.to_csv(path)
    print(f"Table saved: {path}")


print("Setup complete.")

## 2. The building's electricity use

The main example is the **Engineering and Institute for Digital Innovation** building. The cell loads the monthly invoice data for every building and selects this one.

In [ ]:
campus = pd.read_csv(DATA_DIR / "campus_electricity_2021_22.csv")
campus["month"] = pd.PeriodIndex(campus["month"], freq="M")

BUILDING = "Engineering and Institute for Digital Innovation"


def building_demand(name):
    # Monthly electricity use of one building, in kWh, indexed by month.
    rows = campus[campus["building"] == name]
    if rows.empty:
        raise ValueError(f"No building called {name!r}. Check the spelling against campus['building'].unique().")
    return rows.set_index("month")["electricity_kwh"].astype(float)


demand = building_demand(BUILDING)
print(f"{BUILDING}: {len(demand)} months, {demand.sum() / 1000:,.1f} MWh in total")
display(demand.to_frame("Electricity (kWh)").T)

### 2.1 Check the data before you model it

Always look at data before using it. The chart shows the building's monthly electricity use.

**Discuss as a group (one minute):** which months look unusual? List two possible reasons. Think about how buildings are used during the academic year, and about how invoice data is produced.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
labels = demand.index.strftime("%b %y")
ax.bar(labels, demand / 1000, color=COLOURS["demand"])
ax.set_ylabel("Electricity (MWh per month)")
ax.set_title(f"Figure 1. Monthly electricity use: {BUILDING}", loc="left")
plt.show()
save_figure(fig, "task3_fig1_building_demand")

print(f"Highest month: {demand.idxmax().strftime('%B %Y')}, {demand.max() / 1000:.1f} MWh")
print(f"Lowest month:  {demand.idxmin().strftime('%B %Y')}, {demand.min() / 1000:.1f} MWh")
print(f"The highest month is {demand.max() / demand.min():.0f} times the lowest.")

A ratio this large is a warning sign. Possible explanations include low use of the building in early autumn 2021, an estimated rather than measured invoice, or a change of meter. In a real project you would ask the university's energy team before drawing conclusions. Here we continue, but keep it in mind when you interpret the results.

## 3. Rooftop solar output from PVGIS

We use the same source as Task 1: modelled hourly output from **PVGIS** (European Commission Joint Research Centre), here for 1 kWp of south-facing panels at 35° on the same campus, for **the same months as the invoices** (August 2021 to July 2022). The hourly values are added up into monthly totals.

**The roof size is an assumption.** No public figure exists for this roof, so we assume **100 kWp**, roughly 500 m² of panels. You can change `PV_KWP` and run the cells again.

In [ ]:
PV_KWP = 100                  # assumed rooftop system size, kWp
LATITUDE, LONGITUDE = 52.912, -1.184
PVGIS_URL = "https://re.jrc.ec.europa.eu/api/v5_3/seriescalc"
PVGIS_CACHE = DATA_DIR / "pvgis_campus_2021_2022_1kwp.json"


def monthly_pv_per_kwp():
    # Monthly AC output of 1 kWp, in kWh, for August 2021 to July 2022. Downloads once, then uses the saved copy.
    if PVGIS_CACHE.exists():
        print(f"Using the saved PVGIS file: {PVGIS_CACHE}")
        reply = json.loads(PVGIS_CACHE.read_text(encoding="utf-8"))
    else:
        params = {"lat": LATITUDE, "lon": LONGITUDE, "peakpower": 1, "loss": 14, "angle": 35, "aspect": 0,
                  "startyear": 2021, "endyear": 2022, "pvcalculation": 1, "outputformat": "json"}
        print("Downloading hourly solar data from PVGIS ...")
        try:
            response = requests.get(PVGIS_URL, params=params, timeout=60)
            response.raise_for_status()
            reply = response.json()
        except Exception as error:
            raise RuntimeError(f"PVGIS could not be reached ({error}). Ask the instructor for the file "
                               f"{PVGIS_CACHE.name} and put it in the data folder.") from error
        if "outputs" not in reply:
            raise ValueError("PVGIS replied, but without hourly data.")
        PVGIS_CACHE.write_text(json.dumps(reply), encoding="utf-8")
        print(f"Saved to {PVGIS_CACHE}")
    hourly = pd.DataFrame(reply["outputs"]["hourly"])
    times = pd.to_datetime(hourly["time"], format="%Y%m%d:%H%M")
    pv_kwh = pd.Series(hourly["P"].to_numpy() / 1000, index=times)        # W for one hour -> kWh
    monthly = pv_kwh.groupby(pv_kwh.index.to_period("M")).sum()
    return monthly.loc[demand.index]


pv_per_kwp = monthly_pv_per_kwp()
print(f"One kWp produces {pv_per_kwp.sum():,.0f} kWh over these 12 months.")
print(f"A {PV_KWP} kWp system produces {pv_per_kwp.sum() * PV_KWP / 1000:,.1f} MWh.")

## 4. TASK: how much of the solar can the building use?

With monthly data, the simplest assumption is that solar output is used by the building **up to that month's demand**. Anything above the demand is exported to the grid.

**TASK.** Replace the `____` so that `pv_used` is the **smaller** of the solar output and the demand in each month. `np.minimum(a, b)` returns the smaller value, month by month.

In [ ]:
def compare(demand_kwh, pv_kwp):
    # Month-by-month comparison of demand and rooftop solar.
    month_table = pd.DataFrame({
        "Demand (kWh)": demand_kwh,
        "Solar (kWh)": pv_per_kwp * pv_kwp,
    })
    month_table["Solar used (kWh)"] = np.minimum(month_table["Solar (kWh)"], ____)      # TASK
    month_table["Solar exported (kWh)"] = month_table["Solar (kWh)"] - month_table["Solar used (kWh)"]
    month_table["Share of demand from solar (%)"] = 100 * month_table["Solar used (kWh)"] / month_table["Demand (kWh)"]
    return month_table


month_table = compare(demand, PV_KWP)
display(month_table.round(0))
save_table(month_table, "task3_monthly")

If you see `NameError: name '____' is not defined`, the blank is still empty. Fill it in and run the cell again.

The next cell checks your answer, then plots the year.

In [ ]:
ok = np.allclose(month_table["Solar used (kWh)"], np.minimum(month_table["Solar (kWh)"], month_table["Demand (kWh)"]))
print("PASS: solar used is never more than the demand or the solar output." if ok
      else "FAIL: check the blank in section 4.")

fig, ax = plt.subplots(figsize=(9, 3.8))
x = np.arange(len(month_table))
ax.bar(x - 0.2, month_table["Demand (kWh)"] / 1000, width=0.4, color=COLOURS["demand"], label="Demand")
ax.bar(x + 0.2, month_table["Solar used (kWh)"] / 1000, width=0.4, color=COLOURS["used"], label="Solar used")
ax.bar(x + 0.2, month_table["Solar exported (kWh)"] / 1000, width=0.4, bottom=month_table["Solar used (kWh)"] / 1000,
       color=COLOURS["export"], label="Solar exported")
ax.set_xticks(x, month_table.index.strftime("%b %y"))
ax.set_ylabel("MWh per month")
ax.set_title(f"Figure 2. Demand and {PV_KWP} kWp of rooftop solar, month by month", loc="left")
ax.legend(ncols=3, loc="upper left")
plt.show()
save_figure(fig, "task3_fig2_solar_vs_demand")

## 5. Results for the year: solar share and CO2 saved

Each kWh of solar used in the building replaces a kWh bought from the grid. To estimate the CO2 saved we use the UK Government's 2022 greenhouse gas conversion factor for electricity, **0.19338 kg CO2e per kWh** (generation only, excluding transmission and distribution losses). This is an average for the grid over a year, so it is an approximation.

**Predict first:** what share of the building's annual electricity do you expect 100 kWp of solar to supply? Then run the cell.

In [ ]:
GRID_CO2_KG_PER_KWH = 0.19338     # UK Government GHG conversion factors 2022, UK electricity (generation)


def summary(month_table):
    # Annual results for one building and one solar size.
    used = month_table["Solar used (kWh)"].sum()
    return pd.Series({
        "Electricity use (MWh/yr)": month_table["Demand (kWh)"].sum() / 1000,
        "Solar output (MWh/yr)": month_table["Solar (kWh)"].sum() / 1000,
        "Solar used on site (MWh/yr)": used / 1000,
        "Solar exported (MWh/yr)": month_table["Solar exported (kWh)"].sum() / 1000,
        "Share of demand from solar (%)": 100 * used / month_table["Demand (kWh)"].sum(),
        "Months with exported solar": int((month_table["Solar exported (kWh)"] > 0.5).sum()),
        "CO2 saved (tCO2e/yr)": used * GRID_CO2_KG_PER_KWH / 1000,
    }, dtype=object)



def rounded(results):
    # One decimal place for quantities; counts of months stay whole numbers.
    return results.map(lambda v: f"{v:,}" if isinstance(v, int) else f"{float(v):,.1f}")


reference = summary(month_table)
display(rounded(reference).to_frame(f"{BUILDING}, {PV_KWP} kWp"))

## 6. Group scenario: change one thing

Each group tests one change. Type your group number and run the cell. **Before you run it, agree a prediction:** will the share of demand from solar go up or down, and by roughly how much?

| Scenario | What changes |
|---|---|
| U | Solar size 50 kWp |
| V | Solar size 200 kWp |
| W | Solar size 400 kWp, for example including a car-park canopy |
| X | A different building: Clifton Library, with 100 kWp |
| Y | A different building: Interdisciplinary Science and Technology Centre, with 100 kWp |

In [ ]:
MY_GROUP = 1   # TASK: type your group number, then run this cell.

TASK3_SCENARIOS = {
    "U": {"label": "50 kWp", "kwp": 50, "building": BUILDING},
    "V": {"label": "200 kWp", "kwp": 200, "building": BUILDING},
    "W": {"label": "400 kWp", "kwp": 400, "building": BUILDING},
    "X": {"label": "Clifton Library, 100 kWp", "kwp": 100, "building": "Clifton Library"},
    "Y": {"label": "Interdisciplinary Science and Technology Centre, 100 kWp", "kwp": 100,
          "building": "Interdisciplinary Science and Technology Centre"},
}
if not isinstance(MY_GROUP, int) or MY_GROUP < 1:
    raise ValueError(f"MY_GROUP must be a whole number such as 3, not {MY_GROUP!r}.")
letter = "UVWXY"[(MY_GROUP - 1) % 5]
choice = TASK3_SCENARIOS[letter]
print(f"Group {MY_GROUP}: Task 3 scenario {letter}, {choice['label']}\n")

scenario = summary(compare(building_demand(choice["building"]), choice["kwp"]))
table3 = pd.DataFrame({f"Reference: Engineering, {PV_KWP} kWp": reference,
                       f"Scenario {letter}: {choice['label']}": scenario})
display(rounded(table3))
save_table(table3, f"task3_scenario_{letter}")

**Discuss:** did doubling the solar size double the CO2 saved? If not, why not? Which months export solar, and what could the building do with that energy?

### Stretch task: the largest system with no export in any month

Using monthly data, what is the largest solar system whose output never exceeds the building's demand in any month? Run the cell, then compare the answer with the 400 kWp scenario. The cell also repeats the calculation without the limiting month: how much can one doubtful month of data change a design answer?

In [ ]:
largest_kwp = (demand / pv_per_kwp).min()
limiting_month = (demand / pv_per_kwp).idxmin().strftime("%B %Y")
print(f"Largest system with no monthly export: about {largest_kwp:,.0f} kWp, limited by {limiting_month}.")

ratio = (demand / pv_per_kwp).drop((demand / pv_per_kwp).idxmin())
print(f"Without {limiting_month}: about {ratio.min():,.0f} kWp, limited by {ratio.idxmin().strftime('%B %Y')}.")

## 7. What monthly data cannot tell you

Comparing **monthly** totals assumes that solar produced at noon on a Saturday in July can be used on any evening of that month. In reality solar is produced only in daylight, and the building may be quiet at weekends and in the holidays, so some solar is exported even in months where demand is higher than output. **The share of demand from solar calculated here is therefore an upper bound**, and the gap grows with system size.

To size a real system, an engineer would use **half-hourly** meter data, which the university's energy team can usually provide for large buildings, and repeat the hour-by-hour method from Task 1.

**Summary questions for your group:**

1. What share of the building's electricity could 100 kWp of rooftop solar supply, and how much CO2 would it save?
2. Name one reason the real share would be lower, and one piece of data that would improve the estimate.

## References

- European Commission Joint Research Centre. *Photovoltaic Geographical Information System (PVGIS)*, version 5.3, radiation database PVGIS-SARAH3. PVGIS © European Communities, 2001-2026.
- Department for Energy Security and Net Zero and Department for Environment, Food and Rural Affairs (2022). *Greenhouse gas reporting: conversion factors 2022*. UK electricity, generation: 0.19338 kg CO2e per kWh.
- Building electricity data: university energy invoices, August 2021 to July 2022, provided for teaching use only.